In [ ]:
# ==============================
# DLG + iDLG + GradInversion
# EMNIST Digits
# 4 Settings:
# Baseline
# Masking Only
# Masking + ANF=0.1
# Masking + ANF=0.2
# ==============================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import random
import os
from google.colab import files

# -------------------------
# Seeds
# -------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# -------------------------
# Device
# -------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

# -------------------------
# Save Directory
# -------------------------
SAVE_DIR = "./dlg_emnist_digits_4models"

os.makedirs(SAVE_DIR, exist_ok=True)

# -------------------------
# Dataset
# -------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

dataset = torchvision.datasets.EMNIST(
    root="./data",
    split="digits",
    train=True,
    download=True,
    transform=transform
)

# -------------------------
# CNN Model
# -------------------------
class EMNIST_CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)

        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(64 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))

        return self.fc2(x)

# -------------------------
# Load Models
# -------------------------
baseline_model = EMNIST_CNN().to(DEVICE)

fpman_model_0 = EMNIST_CNN().to(DEVICE)
fpman_model_1 = EMNIST_CNN().to(DEVICE)
fpman_model_2 = EMNIST_CNN().to(DEVICE)

baseline_model.load_state_dict(
    torch.load(
        "Baseline_emnist_digits.pth",
        map_location=DEVICE
    )
)

fpman_model_0.load_state_dict(
    torch.load(
        "FPMAN_emnist_digits_0.pth",
        map_location=DEVICE
    )
)

fpman_model_1.load_state_dict(
    torch.load(
        "FPMAN_emnist_digits_1.pth",
        map_location=DEVICE
    )
)

fpman_model_2.load_state_dict(
    torch.load(
        "FPMAN_emnist_digits_2.pth",
        map_location=DEVICE
    )
)

baseline_model.eval()

fpman_model_0.eval()
fpman_model_1.eval()
fpman_model_2.eval()

print("Models loaded")

# -------------------------
# One Sample Per Digit
# -------------------------
digit_samples = {}

for img, label in dataset:

    if label not in digit_samples:
        digit_samples[label] = img

    if len(digit_samples) == 10:
        break

# -------------------------
# Get Gradients
# -------------------------
def get_gradients(model, image, label):

    model.zero_grad()

    pred = model(image)

    loss = F.cross_entropy(pred, label)

    grads = torch.autograd.grad(
        loss,
        model.parameters()
    )

    return grads

# -------------------------
# Total Variation
# -------------------------
def total_variation(x):

    tv_h = (
        (x[:, :, 1:, :] - x[:, :, :-1, :]) ** 2
    ).mean()

    tv_w = (
        (x[:, :, :, 1:] - x[:, :, :, :-1]) ** 2
    ).mean()

    return tv_h + tv_w

# -------------------------
# DLG Attack
# -------------------------
def dlg_attack(
    model,
    original_grads,
    image_shape,
    label,
    steps=400,
    lr=0.05
):

    dummy_data = (
        torch.randn(image_shape, device=DEVICE) * 0.1
    ).detach()

    dummy_data.requires_grad = True

    optimizer = torch.optim.Adam(
        [dummy_data],
        lr=lr
    )

    for step in range(steps):

        optimizer.zero_grad()

        pred = model(dummy_data)

        dummy_loss = F.cross_entropy(
            pred,
            label
        )

        dummy_grads = torch.autograd.grad(
            dummy_loss,
            model.parameters(),
            create_graph=True
        )

        grad_diff = 0

        for gx, gy in zip(dummy_grads, original_grads):

            grad_diff += ((gx - gy) ** 2).sum()

        grad_diff.backward()

        optimizer.step()

        with torch.no_grad():
            dummy_data.clamp_(-1, 1)

    return dummy_data.detach()

# -------------------------
# iDLG Attack
# -------------------------
def idlg_attack(model, original_grads, image_shape, steps=400, lr=0.05):

    # last layer = fc2.bias (shape: [10])
    last_bias_grad = original_grads[-1].detach()

    recovered_label = torch.argmin(last_bias_grad).unsqueeze(0).to(DEVICE)

    dummy_data = (torch.randn(image_shape, device=DEVICE) * 0.1).detach()
    dummy_data.requires_grad = True

    optimizer = torch.optim.Adam([dummy_data], lr=lr)

    for _ in range(steps):

        optimizer.zero_grad()

        pred = model(dummy_data)

        loss = F.cross_entropy(pred, recovered_label)

        dummy_grads = torch.autograd.grad(
            loss,
            model.parameters(),
            create_graph=True
        )

        grad_diff = 0
        for gx, gy in zip(dummy_grads, original_grads):
            grad_diff += ((gx - gy) ** 2).sum()

        grad_diff.backward()
        optimizer.step()

        with torch.no_grad():
            dummy_data.clamp_(-1, 1)

    return dummy_data.detach()

# -------------------------
# GradInversion Attack
# -------------------------
def gradinv_attack(
    model,
    original_grads,
    image_shape,
    label,
    steps=400,
    lr=0.05,
    tv_weight=1e-4
):

    dummy_data = (
        torch.randn(image_shape, device=DEVICE) * 0.1
    ).detach()

    dummy_data.requires_grad = True

    optimizer = torch.optim.Adam(
        [dummy_data],
        lr=lr
    )

    for step in range(steps):

        optimizer.zero_grad()

        pred = model(dummy_data)

        dummy_loss = F.cross_entropy(
            pred,
            label
        )

        dummy_grads = torch.autograd.grad(
            dummy_loss,
            model.parameters(),
            create_graph=True
        )

        grad_diff = 0

        for gx, gy in zip(dummy_grads, original_grads):

            grad_diff += ((gx - gy) ** 2).sum()

        grad_diff += (
            tv_weight *
            total_variation(dummy_data)
        )

        grad_diff.backward()

        optimizer.step()

        with torch.no_grad():
            dummy_data.clamp_(-1, 1)

    return dummy_data.detach()

# -------------------------
# Models Dictionary
# -------------------------
models = {
    "baseline": baseline_model,
    "fpman0": fpman_model_0,
    "fpman1": fpman_model_1,
    "fpman2": fpman_model_2
}

# -------------------------
# Results Storage
# -------------------------
results = {
    k: {
        "dlg": [],
        "idlg": [],
        "gi": []
    }
    for k in models
}

original_imgs = []

# -------------------------
# Run Attacks
# -------------------------
for digit in range(10):

    print(f"\nRunning Digit: {digit}")

    img = (
        digit_samples[digit]
        .unsqueeze(0)
        .to(DEVICE)
    )

    label = torch.tensor([digit]).to(DEVICE)

    original_imgs.append(
        img.cpu().numpy()[0][0]
    )

    for name, model in models.items():

        print(f"  Model: {name}")

        grads = get_gradients(
            model,
            img,
            label
        )

        # -----------------
        # DLG
        # -----------------
        print("    DLG...")

        recon_dlg = dlg_attack(
            model,
            grads,
            img.shape,
            label
        )

        results[name]["dlg"].append(
            recon_dlg.cpu().numpy()[0][0]
        )

        # -----------------
        # iDLG
        # -----------------
        print("    iDLG...")

        recon_idlg = idlg_attack(
            model,
            grads,
            img.shape
        )

        results[name]["idlg"].append(
            recon_idlg.cpu().numpy()[0][0]
        )

        # -----------------
        # GradInversion
        # -----------------
        print("    GradInversion...")

        recon_gi = gradinv_attack(
            model,
            grads,
            img.shape,
            label
        )

        results[name]["gi"].append(
            recon_gi.cpu().numpy()[0][0]
        )

# -------------------------
# Visualization
# -------------------------
row_names = [

    "Original",

    "Baseline DLG",
    "Baseline iDLG",
    "Baseline GradInv",

    "FPMAN0 DLG",
    "FPMAN0 iDLG",
    "FPMAN0 GradInv",

    "FPMAN1 DLG",
    "FPMAN1 iDLG",
    "FPMAN1 GradInv",

    "FPMAN2 DLG",
    "FPMAN2 iDLG",
    "FPMAN2 GradInv"
]

all_rows = [

    original_imgs,

    results["baseline"]["dlg"],
    results["baseline"]["idlg"],
    results["baseline"]["gi"],

    results["fpman0"]["dlg"],
    results["fpman0"]["idlg"],
    results["fpman0"]["gi"],

    results["fpman1"]["dlg"],
    results["fpman1"]["idlg"],
    results["fpman1"]["gi"],

    results["fpman2"]["dlg"],
    results["fpman2"]["idlg"],
    results["fpman2"]["gi"]
]

fig, axes = plt.subplots(
    len(all_rows),
    10,
    figsize=(22, 22)
)

for r in range(len(all_rows)):

    for c in range(10):

        axes[r, c].imshow(
            all_rows[r][c],
            cmap="gray"
        )

        axes[r, c].axis("off")

# -------------------------
# Column Titles
# -------------------------
for c in range(10):

    axes[0, c].set_title(
        f"Digit {c}",
        fontsize=12
    )

# -------------------------
# Row Labels
# -------------------------
for i, name in enumerate(row_names):

    fig.text(
        0.02,
        0.95 - i * (0.9 / len(row_names)),
        name,
        fontsize=10,
        rotation=90,
        va="center"
    )

# -------------------------
# Main Title
# -------------------------
plt.suptitle(
    "EMNIST Digits Privacy Leakage Comparison\n"
    "Baseline vs Masking vs ANF=0.1 vs ANF=0.2",
    fontsize=18
)

plt.tight_layout(
    rect=[0.05, 0, 1, 0.96]
)

# -------------------------
# Save Figure
# -------------------------
save_path = os.path.join(
    SAVE_DIR,
    "dlg_idlg_gradinv_emnist_digits_4models.png"
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:", save_path)

# -------------------------
# Download
# -------------------------
files.download(save_path)

Device: cpu
Models loaded

Running Digit: 0
  Model: baseline
    DLG...


In [ ]:
# -------------------------
# Visualization: 13 Separate Images with Thicker Margin (EMNIST Digits / 4 Models)
# -------------------------

row_names = [
    "original",

    "baseline_dlg",
    "baseline_idlg",
    "baseline_gradinv",

    "fpman0_dlg",
    "fpman0_idlg",
    "fpman0_gradinv",

    "fpman1_dlg",
    "fpman1_idlg",
    "fpman1_gradinv",

    "fpman2_dlg",
    "fpman2_idlg",
    "fpman2_gradinv"
]

all_rows = [
    original_imgs,

    results["baseline"]["dlg"],
    results["baseline"]["idlg"],
    results["baseline"]["gi"],

    results["fpman0"]["dlg"],
    results["fpman0"]["idlg"],
    results["fpman0"]["gi"],

    results["fpman1"]["dlg"],
    results["fpman1"]["idlg"],
    results["fpman1"]["gi"],

    results["fpman2"]["dlg"],
    results["fpman2"]["idlg"],
    results["fpman2"]["gi"],
]

for idx, row in enumerate(all_rows):

    fig, axes = plt.subplots(1, 10, figsize=(22, 3))

    for c in range(10):
        axes[c].imshow(row[c], cmap="gray")
        axes[c].axis("off")

    save_path = os.path.join(SAVE_DIR, f"{row_names[idx]}.png")

    plt.savefig(save_path, dpi=300, bbox_inches='tight', pad_inches=0.2)
    plt.show()

    print("Figure saved:", save_path)

    files.download(save_path)